In [ ]:
import pandas as pd
import numpy as np
import joblib
import pickle
import xgboost as xgb
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings

In [ ]:
# Ignore harmless warnings from statsmodels
warnings.filterwarnings("ignore")

In [ ]:
# ==========================================
# 1. Read and preprocess the test data
# ==========================================
print("Reading test data...")
try:
    df = pd.read_csv('../data/BTCUSD1D-test.csv')
except FileNotFoundError:
    print("Error: File 'BTCUSD1D-test.csv' not found.")
    exit()

In [ ]:
# Convert datetime column to datetime format and sort chronologically
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values('datetime')
df = df.set_index('datetime')

In [ ]:
# Keep a copy of the full 'close' series for SARIMA before dropping any rows
full_close_series = df['close'].copy()

In [ ]:
# ==========================================
# 2. Feature Engineering for LR and XGBoost
# ==========================================
window_size = 30
print(f"Creating {window_size} lag features...")

In [ ]:
for i in range(1, window_size + 1):
    df[f'Lag_{i}'] = df['close'].shift(i)

In [ ]:
# Drop the first 30 rows that now contain missing data (NaNs) due to shifting
df_lagged = df.dropna()

In [ ]:
# Extract Features (X) and Actual Prices (y)
feature_cols = [f'Lag_{i}' for i in range(1, window_size + 1)]
X = df_lagged[feature_cols]
actual_prices = df_lagged['close']

In [ ]:
# The 'Lag_1' column represents the actual close price of the previous session
prev_actual_prices = X['Lag_1']

In [ ]:
# ==========================================
# 3. Load the pre-trained models
# ==========================================
print("Loading pre-trained models...")

In [ ]:
# Load Linear Regression
lr_model = joblib.load('models/1_lr_btc_30days_model.pkl')

In [ ]:
# Load SARIMA
with open('models/2_sarima_rolling_btc_model.pkl', 'rb') as f:
    sarima_loaded = pickle.load(f)

In [ ]:
# Load XGBoost
xgb_model = xgb.XGBRegressor()
xgb_model.load_model('models/3_xgboost_btc_30days_model.json')

In [ ]:
# ==========================================
# 4. Generate Absolute Price Predictions
# ==========================================
print("Generating 1-step ahead price predictions...")

In [ ]:
# Linear Regression predictions
lr_preds = lr_model.predict(X)

In [ ]:
# XGBoost predictions
xgb_preds = xgb_model.predict(X)

In [ ]:
# SARIMA predictions (Rolling Filter to use actual past data)
sarima_structure = SARIMAX(full_close_series, order=(1, 1, 1), seasonal_order=(0, 0, 0, 0), enforce_stationarity=False, enforce_invertibility=False)
sarima_filtered = sarima_structure.filter(sarima_loaded.params)
sarima_full_preds = sarima_filtered.predict(dynamic=False)

In [ ]:
# Align SARIMA predictions with the lagged dataset's dates
sarima_preds = sarima_full_preds.loc[actual_prices.index].values

In [ ]:
# ==========================================
# 5. Convert Prices to UP/DOWN Trends
# ==========================================
print("Calculating UP/DOWN trends...")

In [ ]:
# Function to determine trend: UP if current > previous, else DOWN
def get_trend(current_values, prev_values):
    return np.where(current_values > prev_values, 'UP', 'DOWN')

In [ ]:
actual_trend = get_trend(actual_prices, prev_actual_prices)
lr_trend = get_trend(lr_preds, prev_actual_prices)
sarima_trend = get_trend(sarima_preds, prev_actual_prices)
xgb_trend = get_trend(xgb_preds, prev_actual_prices)

In [ ]:
# ==========================================
# 6. Calculate Average Trend (Majority Vote)
# ==========================================
# Convert UP to 1 and DOWN to 0 for easy mathematical voting
def trend_to_numeric(trend_array):
    return np.where(trend_array == 'UP', 1, 0)

In [ ]:
lr_num = trend_to_numeric(lr_trend)
sarima_num = trend_to_numeric(sarima_trend)
xgb_num = trend_to_numeric(xgb_trend)

In [ ]:
# Sum the votes. If sum >= 2, it means at least 2 out of 3 models predicted UP.
majority_vote_sum = lr_num + sarima_num + xgb_num
average_trend = np.where(majority_vote_sum >= 2, 'UP', 'DOWN')

In [ ]:
# ==========================================
# 7. Prepare DataFrame and Print Results
# ==========================================
results_df = pd.DataFrame({
    'Actual Price': actual_prices.values,
    'LR Predict': lr_preds,
    'SARIMA Predict': sarima_preds,
    'XGB Predict': xgb_preds,
    'Actual Trend': actual_trend,
    'LR Trend Predict': lr_trend,
    'SARIMA Trend Predict': sarima_trend,
    'XGB Trend Predict': xgb_trend,
    'Average Trend Predict': average_trend
}, index=actual_prices.index)

In [ ]:
# Print specific columns to the terminal as requested
print("\n--- Trend Prediction Results ---")
display_cols = ['Actual Price', 'Actual Trend', 'LR Trend Predict', 'SARIMA Trend Predict', 'XGB Trend Predict', 'Average Trend Predict']
print(results_df[display_cols].head(20))

In [ ]:
# ==========================================
# 8. Export to CSV
# ==========================================
csv_filename = 'csv/5_average_trend_predict.csv'
# Reorder columns specifically for the CSV output to maintain clarity and match requested data structure
csv_export_cols = [
    'Actual Price', 'LR Predict', 'SARIMA Predict', 'XGB Predict', 
    'Actual Trend', 'LR Trend Predict', 'SARIMA Trend Predict', 
    'XGB Trend Predict', 'Average Trend Predict'
]

In [ ]:
results_df[csv_export_cols].to_csv(csv_filename, index=True)
print(f"\nSuccessfully exported full results to: '{csv_filename}'")